Upload Dataset to Colab

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving archive (4).zip to archive (4).zip


Extract ZIP File

In [2]:
import zipfile

with zipfile.ZipFile('movie_genre.zip', 'r') as zip_ref:
    zip_ref.extractall('movie_dataset')

Verify Files

In [4]:
import os

for root, dirs, files in os.walk('movie_dataset'):
    print(files)

[]
['test_data_solution.txt', 'description.txt', 'test_data.txt', 'train_data.txt']


In [6]:
!find /content -name "train_data.txt"

/content/movie_dataset/Genre Classification Dataset/train_data.txt


Read First Record

In [7]:
with open('/content/movie_dataset/Genre Classification Dataset/train_data.txt',
          'r',
          encoding='utf-8') as f:

    lines = f.readlines()

print(lines[0])

1 ::: Oscar et la dame rose (2009) ::: drama ::: Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.



Import Libraries

In [8]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

Load Training Data

In [9]:
train_path = "/content/movie_dataset/Genre Classification Dataset/train_data.txt"

df = pd.read_csv(
    train_path,
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID","TITLE","GENRE","DESCRIPTION"]
)

df.head()

,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


Display Dataset Information

In [10]:
print(df.shape)
print(df.columns)

(54214, 4)
Index(['ID', 'TITLE', 'GENRE', 'DESCRIPTION'], dtype='object')


Check Missing Values

In [11]:
print(df.isnull().sum())

ID             0
TITLE          0
GENRE          0
DESCRIPTION    0
dtype: int64


Select Features and Labels

In [12]:
X = df["DESCRIPTION"]

y = df["GENRE"]

TF-IDF Vectorization

In [13]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X = tfidf.fit_transform(X)

Split Dataset

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Train Logistic Regression Model

In [15]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Predictions

In [16]:
predictions = model.predict(X_test)

Accuracy

In [17]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy*100,2), "%")

Accuracy: 58.01 %


Classification Report

In [18]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

      action       0.52      0.26      0.35       263
       adult       0.75      0.21      0.33       112
   adventure       0.43      0.14      0.22       139
   animation       0.60      0.09      0.15       104
   biography       0.00      0.00      0.00        61
      comedy       0.52      0.59      0.55      1443
       crime       0.29      0.02      0.04       107
 documentary       0.66      0.84      0.74      2659
       drama       0.54      0.78      0.64      2697
      family       0.41      0.07      0.12       150
     fantasy       0.00      0.00      0.00        74
   game-show       0.94      0.42      0.59        40
     history       0.00      0.00      0.00        45
      horror       0.64      0.56      0.60       431
       music       0.63      0.47      0.53       144
     musical       0.50      0.02      0.04        50
     mystery       0.00      0.00      0.00        56
        news       1.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Genre Prediction Function

In [19]:
def predict_genre(text):

    vector = tfidf.transform([text])

    prediction = model.predict(vector)

    return prediction[0]

Test Prediction

In [20]:
movie = """
A young warrior fights against evil forces
to save his kingdom and protect his people.
"""

print("Predicted Genre:", predict_genre(movie))

Predicted Genre: action


Save Model

In [21]:
import pickle

pickle.dump(model, open("movie_genre_model.pkl","wb"))
pickle.dump(tfidf, open("tfidf.pkl","wb"))

print("Model Saved Successfully")

Model Saved Successfully
